In [6]:
#!/usr/bin/env python3
"""
Build a long-format reverse-lookup mismatch table for classifier outputs.

Inputs:
  - X_dev.csv (must contain TokenID)
  - y_dev.csv (gold labels; columns per language)
  - y_pred_*.csv (predicted labels; columns per language)

Output:
  - CSV with columns: TokenID, lang, gold, pred (only mismatches)
"""

from __future__ import annotations
import argparse
import sys
from typing import Sequence, List
import pandas as pd


def build_mismatch_table(
    X_dev_path: str,
    y_dev_path: str,
    y_pred_path: str,
    output_csv_path: str,
    lang_cols: Sequence[str] = ("eng_constr", "deu_constr", "xho_constr", "yor_constr"),
    token_id_col: str = "TokenID",
) -> pd.DataFrame:
    # Load inputs
    X_dev = pd.read_csv(X_dev_path)
    y_dev = pd.read_csv(y_dev_path)
    y_pred = pd.read_csv(y_pred_path)

    # Tolerate stray index columns like 'Unnamed: 0'
    for df in (y_dev, y_pred):
        if "Unnamed: 0" in df.columns and token_id_col not in df.columns:
            df = df.drop(columns=["Unnamed: 0"])

    # Basic checks
    if len(X_dev) != len(y_dev) or len(y_dev) != len(y_pred):
        raise ValueError(
            f"Row count mismatch: X_dev={len(X_dev)}, y_dev={len(y_dev)}, y_pred={len(y_pred)}"
        )
    if token_id_col not in X_dev.columns:
        raise ValueError(f"'{token_id_col}' not found in X_dev columns: {list(X_dev.columns)}")

    for c in lang_cols:
        if c not in y_dev.columns:
            raise ValueError(f"'{c}' missing from y_dev columns: {list(y_dev.columns)}")
        if c not in y_pred.columns:
            raise ValueError(f"'{c}' missing from y_pred columns: {list(y_pred.columns)}")

    # Pull TokenIDs (aligned by row order)
    token_ids = X_dev[token_id_col].reset_index(drop=True)

    # Build long-format mismatches
    records: List[pd.DataFrame] = []
    for lang_col in lang_cols:
        gold = y_dev[lang_col].reset_index(drop=True)
        pred = y_pred[lang_col].reset_index(drop=True)
        lang_code = lang_col.split("_")[0].upper()  # 'eng_constr' -> 'ENG'

        mism_mask = gold != pred
        if mism_mask.any():
            subset = pd.DataFrame(
                {
                    "TokenID": token_ids[mism_mask].values,
                    "lang": lang_code,
                    "gold": gold[mism_mask].values,
                    "pred": pred[mism_mask].values,
                }
            )
            records.append(subset)

    mismatches = (
        pd.concat(records, ignore_index=True)
        if records
        else pd.DataFrame(columns=["TokenID", "lang", "gold", "pred"])
    )

    mismatches = mismatches.sort_values(["lang", "TokenID"]).reset_index(drop=True)
    mismatches.to_csv(output_csv_path, index=False)
    return mismatches

def main() -> None:
    args = parse_args()
    try:
        build_mismatch_table(
            X_dev_path=args.x_dev,
            y_dev_path=args.y_dev,
            y_pred_path=args.y_pred,
            output_csv_path=args.out,
            lang_cols=args.langs,
            token_id_col=args.token_col,
        )
    except Exception as e:
        print(f"[ERROR] {e}", file=sys.stderr)
        sys.exit(1)


#if __name__ == "__main__":
#    main()

In [7]:
# Example call inside Jupyter:
mismatches = build_mismatch_table(
    X_dev_path="X_dev.csv",
    y_dev_path="y_dev.csv",
    y_pred_path="y_pred_mlp.csv",
    output_csv_path="mismatches_mlp_long.csv"
)

# Peek at the first few rows
mismatches.head()

,TokenID,lang,gold,pred
0,1022019,DEU,n_a,pastperf
1,1022020,DEU,n_a,perf
2,1020005,ENG,perf,pastperf
3,1022020,ENG,pastperf,perf
4,1001031,XHO,ipv_rel,pfv_ind
